# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all Record Sets in the dataset, their @id and field @id's
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets in dataset: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    {f['@id']}")
            else:
                print(f"    {f}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Typically, for mass spec datasets, there is one main record set.
# Let's automatically extract all data into DataFrames and list columns for each.
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs['@id']))
    dataframes[rs['@id']] = pd.DataFrame(records)
    print(f"Loaded Record Set: {rs['@id']}, DataFrame shape: {dataframes[rs['@id']].shape}")
    print(f"Columns: {dataframes[rs['@id']].columns.tolist()}\n")

# Choose the first record set for EDA if present
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"Selected main record set for analysis: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found - check dataset definition.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Analyze a numeric field such as 'cr:field:MW' (Molecular Weight) if present.
df = dataframes[main_record_set_id]

# Try to pick a numeric field for demonstration (fallback to 'MW' or first float/int type)
numeric_field = None
for col in df.columns:
    if df[col].dtype in ['float64', 'int64'] or (df[col].dtype == 'O' and pd.api.types.is_numeric_dtype(pd.to_numeric(df[col], errors='coerce'))):
        numeric_field = col
        break
if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    # Convert column to numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (choose the first object type not numeric)
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'O' and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field, dropna=True).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualize histogram of the main numeric field
if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30, color='steelblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a normalized version exists, plot comparison
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=30, color='orange')
        plt.title(f"Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

# If two or more numeric fields, make a scatterplot
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if len(numeric_cols) >= 2:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
    plt.title(f"{numeric_cols[0]} vs {numeric_cols[1]}")
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`, previewed record sets and their fields via their `@id`s, and extracted records for data analysis. Using basic filtering, normalization, grouping, and visualization techniques, you can quickly start biological or machine learning exploration of protein abundance and modifications. For deeper analysis, consult the Croissant metadata for semantic details, or use domain knowledge relevant to proteomics.